# 5.2 · Implementación I: esqueleto y herramientas

**Hito:** el agente procesa un correo y devuelve texto (todavía sin subagente ni JSON).

Las secciones siguen la pizarra del 5.1.

## 1. Preparación

`imprimir()` y `restaurar_datos()` vienen de `taller_utils.py`.

In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv()

import sys; sys.path.append("../..")   # para importar desde la raíz del taller
from pathlib import Path
from config import MODEL

import json
import herramientas
from claude_agent_sdk import query, tool, create_sdk_mcp_server, ClaudeAgentOptions, ToolAnnotations
from taller_utils import imprimir, restaurar_datos

restaurar_datos()

## 2. Herramientas

Antes de cada descripción, una pregunta: **¿qué necesita saber el modelo para decidir cuándo usar esto?**

In [ ]:
def texto(dato):
    if not isinstance(dato, str):
        dato = json.dumps(dato, ensure_ascii=False)
    return {"content": [{"type": "text", "text": dato}]}

def error(mensaje):
    return {"content": [{"type": "text", "text": mensaje}], "is_error": True}

`leer_correo`: qué recibe (un nombre de fichero, con un ejemplo del formato) y qué devuelve. Si el fichero no existe, el error lista los correos disponibles.

In [ ]:
@tool("leer_correo",
      "Lee un correo de la bandeja de soporte a partir de su nombre de fichero "
      "(p. ej. '001_exportar_excel.txt'). Devuelve el texto completo: remitente, asunto y cuerpo.",
      {"nombre": str},
      annotations=ToolAnnotations(readOnlyHint=True))
async def leer_correo(args):
    try:
        return texto(herramientas.leer_correo(args["nombre"]))
    except FileNotFoundError as e:
        return error(str(e))

`buscar_incidencias`: **cuándo** (antes de crear cualquier incidencia), cómo se combinan los filtros y qué devuelve si no hay nada.

In [ ]:
@tool("buscar_incidencias",
      "Busca incidencias de soporte existentes. Úsala antes de crear cualquier incidencia, para "
      "comprobar si ya hay una abierta sobre el mismo problema. Todos los filtros son opcionales y se "
      "combinan. Devuelve una lista JSON con id, cliente, asunto, estado, prioridad y fecha; [] si no "
      "hay coincidencias.",
      {
          "type": "object",
          "properties": {
              "texto": {"type": "string", "description": "Palabras clave a buscar en asunto y descripción"},
              "estado": {"type": "string", "enum": ["abierta", "cerrada"]},
              "cliente": {"type": "string", "description": "Nombre comercial corto del cliente, p. ej. 'ACME'"},
          },
      },
      annotations=ToolAnnotations(readOnlyHint=True))
async def buscar_incidencias(args):
    return texto(herramientas.buscar_incidencias(**args))

`crear_incidencia`: **cuándo sí y cuándo no**.

In [ ]:
@tool("crear_incidencia",
      "Crea una incidencia nueva y devuelve la incidencia creada con su id. Úsala solo cuando el "
      "correo describe un problema y no existe ya una incidencia abierta sobre él para ese cliente.",
      {
          "type": "object",
          "properties": {
              "cliente": {"type": "string", "description": "Nombre comercial corto del cliente"},
              "asunto": {"type": "string", "description": "Resumen en menos de 10 palabras"},
              "descripcion": {"type": "string", "description": "Qué ocurre, desde cuándo y a quién afecta"},
              "prioridad": {"type": "string", "enum": ["alta", "media", "baja"]},
          },
          "required": ["cliente", "asunto", "descripcion", "prioridad"],
      })
async def crear_incidencia(args):
    return texto(herramientas.crear_incidencia(**args))

Cada herramienta, probada sola antes de seguir:

In [ ]:
await leer_correo.handler({"nombre": "001_exportar_excel.txt"})

In [ ]:
await leer_correo.handler({"nombre": "no_existe.txt"})

In [ ]:
await buscar_incidencias.handler({"cliente": "Umbrella", "estado": "abierta"})

## 3. System prompt y opciones

El system prompt son los pasos del triaje, tal como los dibujamos en la pizarra. La última frase recuerda el alcance, pero el alcance de verdad lo imponen las herramientas: no hay ninguna para enviar ni para cerrar.

In [ ]:
SYSTEM = """Eres un asistente de triaje de soporte. Para el correo indicado:
1. Determina si describe un problema real. Si no, no crees nada.
2. Identifica al cliente y busca si ya existe una incidencia abierta sobre el mismo problema.
3. Si no existe, créala.
4. Busca en la base de conocimiento un artículo que ayude y propón una respuesta basada SOLO en él.
Nunca envías nada: tu salida la revisa una persona."""

In [ ]:
soporte = create_sdk_mcp_server(name="soporte", version="1.0.0",
                                tools=[leer_correo, buscar_incidencias, crear_incidencia])

options = ClaudeAgentOptions(
    model=MODEL, system_prompt=SYSTEM, setting_sources=[],
    mcp_servers={"soporte": soporte},
    tools=[],
    allowed_tools=["mcp__soporte__leer_correo", "mcp__soporte__buscar_incidencias",
                   "mcp__soporte__crear_incidencia"],
    max_turns=15,
)

Una `query()` por correo: cada correo es una tarea independiente, sin historial compartido.

In [ ]:
async def triar(nombre_correo):
    async for msg in query(prompt=f"Criba el correo {nombre_correo}", options=options):
        imprimir(msg)

## 4. Probar con un correo

Uno sencillo:

In [ ]:
print(herramientas.leer_correo("001_exportar_excel.txt"))

In [ ]:
await triar("001_exportar_excel.txt")

Lee el correo, busca, crea la incidencia… y en el paso 4 no puede buscar en la base de conocimiento, porque no tiene la herramienta. Ese es el reto.

In [ ]:
restaurar_datos()

## 5. Procesar la bandeja

Procesar los cinco correos de la bandeja, con salida estructurada y trazas, queda para el 5.3.

## 🔸 Reto

Añade la herramienta `buscar_kb` (está en `taller_utils.py`, del reto del 4.1) o conecta el MCP de ficheros del 4.2, y comprueba que el agente cita un artículo al triar `001_exportar_excel.txt`.